<a href="https://colab.research.google.com/github/AKookani/BrickwallCliffordCircuit/blob/main/Python_translation_of_NN_for_Measurement_Induced_Phase_Transitions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install stim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 32.4 MB/s eta 0:00:00


In [2]:
import numpy as np
import stim

def return_pauli(p, sign=0x0):
    # Converts a binary vector p into a PauliOperator:
    # first half = X components, second half = Z components, with optional phase sign.
    p = np.asarray(p, dtype=bool)
    n = len(p)
    half = n // 2
    x_part = p[:half]
    z_part = p[half:]

    # Build Pauli string: combine X and Z parts per qubit
    # X=1, Z=2, Y=3 (X+Z), I=0
    pauli_chars = []
    for x, z in zip(x_part, z_part):
        if x and z:
            pauli_chars.append('Y')
        elif x:
            pauli_chars.append('X')
        elif z:
            pauli_chars.append('Z')
        else:
            pauli_chars.append('I')

    # Map sign byte to stim's phase convention (0x0=+1, 0x1=+i, 0x2=-1, 0x3=-i)
    sign_map = {0x0: '+', 0x1: '+', 0x2: '-', 0x3: '-'}
    sign_char = sign_map.get(sign, '+')

    pauli_str = sign_char + ''.join(pauli_chars)
    return stim.PauliString(pauli_str)

## X part and Z part — The Symplectic Representation

In quantum computing, every Pauli operator is built from these 4 single-qubit matrices:

$$I = \begin{pmatrix}1&0\\0&1\end{pmatrix}, \quad X = \begin{pmatrix}0&1\\1&0\end{pmatrix}, \quad Z = \begin{pmatrix}1&0\\0&-1\end{pmatrix}, \quad Y = iXZ = \begin{pmatrix}0&-i\\i&0\end{pmatrix}$$

Instead of storing these matrices directly, we store them as **two bits per qubit**:

```
Does this qubit have an X?  → x bit (0 or 1)
Does this qubit have a Z?   → z bit (0 or 1)
```

That's it. Y is just "has both X and Z".

---

### Concrete Example

Say you have a 3-qubit system with operator **X ⊗ Y ⊗ Z** (⊗ means "applied to each qubit"):

| Qubit | Pauli | Has X? | Has Z? |
|-------|-------|--------|--------|
| 0 | X | 1 | 0 |
| 1 | Y | 1 | 1 |
| 2 | Z | 0 | 1 |

So the flat binary vector `p` is:

```
p = [ 1, 1, 0,   0, 1, 1 ]
      ↑  ↑  ↑    ↑  ↑  ↑
      X-part     Z-part
    (q0,q1,q2) (q0,q1,q2)
```
---

### Why this representation?
This is called the **symplectic representation** and it's used heavily in **stabilizer/Clifford** circuit simulation (which is what `QuantumClifford.jl` and `stim` are built for) because:
- It's **memory efficient** — just bits, no complex matrices
- Pauli multiplication becomes simple **bitwise XOR**
- It scales to thousands of qubits easily

## The Sign (Phase) of a Pauli Operator

### Why does a Pauli operator need a sign?

When you multiply Pauli matrices together, you don't always get a clean result — you pick up **phase factors**. For example:

$$X \cdot Z = -iY \qquad Z \cdot X = +iY$$

So the same set of X/Z bits can represent **4 different operators** depending on the phase. That's why the sign is stored separately.

### Concrete Example

Take the X/Z bits for a single qubit Y operator (`x=1, z=1`). Depending on the sign, you get 4 different operators:

| `sign` | Full operator |
|--------|--------------|
| `0x0` | +Y |
| `0x1` | +iY |
| `0x2` | −Y |
| `0x3` | −iY |

---

### When does the sign actually matter?

**1. Stabilizer states** — In stabilizer formalism, qubits are defined by their stabilizers. The sign tells you *which eigenstate* you're in:

```
+Z  stabilizes  |0⟩   (eigenvalue +1)
-Z  stabilizes  |1⟩   (eigenvalue -1)
```
So `+Z` and `−Z` describe **completely different physical states**, even though the X/Z bits are identical.

**2. Pauli multiplication** — When you multiply two Pauli operators, the signs multiply too (and can accumulate factors of ±i). Tracking this correctly is essential for simulation.

**3. Measurement outcomes** — When you measure a Pauli operator, the sign flips tells you whether the result is +1 or −1, which corresponds to the **physical 0 or 1 outcome** of the measurement.


In [3]:
p = [1, 0, 1, 0, 1, 0]  # n=3 qubits: X=[1,0,1], Z=[0,1,0]
return_pauli(p, sign=0x0)
# → +XZX   (stim.PauliString)

stim.PauliString("+XZX")

In [7]:
import numpy as np
import urllib.request

def RandUnitaries():
    # Reads 2-qubit Clifford unitaries from a text file and reorders them into
    # a symplectic tableau format: each group of 4 rows is rearranged as
    # [Z1, X1, X2, Z2], then columns 1 and 4 are swapped.

    Nline = 2880
    unitaries = np.zeros((Nline, 4), dtype=int)

    # Use the raw GitHub URL to fetch the actual text file content
    url = "https://raw.githubusercontent.com/AKookani/BrickwallCliffordCircuit/main/twoUnitaries.txt"

    # Add a User-Agent header to prevent GitHub from blocking the request (403 Forbidden)
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})

    with urllib.request.urlopen(req) as response:
        # Decode the response from bytes to string and split into lines
        lines = response.read().decode('utf-8').splitlines()

        line_idx = 0
        for s in lines:
            s = s.strip()
            if not s:
                continue  # Skip empty lines if any exist

            unitaries[line_idx, 0] = int(s[0])
            unitaries[line_idx, 1] = int(s[2])
            unitaries[line_idx, 2] = int(s[4])
            unitaries[line_idx, 3] = int(s[6])
            line_idx += 1

    randU = np.zeros((Nline, 4), dtype=int)

    for n in range(Nline // 4):
        block = unitaries[4*n : 4*n+4, :]         # shape (4, 4)
        randU[4*n + 0, :] = block[:, 2]  # Z1 row
        randU[4*n + 1, :] = block[:, 0]  # X1 row
        randU[4*n + 3, :] = block[:, 3]  # Z2 row
        randU[4*n + 2, :] = block[:, 1]  # X2 row

    # Swap columns 0 and 3
    randU[:, [0, 3]] = randU[:, [3, 0]]

    return randU

It parses `twoUnitaries.txt`[Link to the .txt file](https://github.com/AKookani/BrickwallCliffordCircuit/blob/main/twoUnitaries.txt), which stores 2-qubit Clifford unitaries as rows of 4 integers (space-separated characters). The 2880 rows are grouped in blocks of 4, each representing one unitary's symplectic tableau. The rows within each block are reordered from the file's column order `[col1, col2, col3, col4]` into the row arrangement `[Z1, X1, X2, Z2]`. Finally, the first and last columns of the entire matrix are swapped, likely to match a specific symplectic convention used downstream.

## `RandUnitaries()` — Overview

The function loads and reformats a table of 2-qubit Clifford unitaries for use in a symplectic (Pauli) tableau framework.

**1. Load raw data**
Reads 2880 lines from `twoUnitaries.txt`, parsing 4 integer values per line (space-separated characters) into a matrix. Each row represents one element of a unitary's symplectic representation.

**2. Reorder into symplectic tableau convention**
The data arrives in groups of 4 rows per unitary. Each group is reordered so the rows follow the convention `[Z1, X1, X2, Z2]` — matching the standard ordering of Pauli generators in a symplectic tableau (X and Z stabilizers for each qubit).

**3. Swap columns 0 and 3**
Swaps the first and last columns across all rows, likely reconciling a difference between the file's qubit ordering and the expected internal convention (e.g., qubit 1 ↔ qubit 2, or bit ordering within the symplectic vector).

**Net result:** Returns `randU`, a reformatted matrix of 2-qubit Clifford unitaries ready for use in stabilizer/symplectic circuit simulations (e.g., randomized benchmarking or random Clifford circuit sampling).

In [8]:
result = RandUnitaries()
print(f"Shape of result: {result.shape}")
print(f"Data type: {result.dtype}")
print("\nFirst 10 rows:")
print(result[:10])
print("\nLast 10 rows:")
print(result[-10:])
print(f"\nUnique values: {np.unique(result)}")

Shape of result: (2880, 4)
Data type: int64

First 10 rows:
[[0 0 0 1]
 [0 0 1 0]
 [1 0 0 0]
 [0 1 0 0]
 [0 0 0 1]
 [0 0 1 0]
 [1 1 0 0]
 [0 1 0 0]
 [0 0 0 1]
 [0 0 1 0]]

Last 10 rows:
[[0 1 1 1]
 [1 1 1 1]
 [0 0 1 1]
 [0 1 1 0]
 [1 1 1 1]
 [1 0 1 1]
 [0 0 1 1]
 [0 1 1 0]
 [1 0 1 1]
 [1 1 1 1]]

Unique values: [0 1]
